# 01 Data generation

## Company brief

CartPadi Nigeria Ltd. is a fictional pan-Nigerian B2C e-commerce marketplace. It sells electronics, phones, fashion, beauty, home and sports products, ships to 24 states, and takes card, bank transfer, USSD, pay on delivery and its own CartPadi wallet. Lagos and Abuja are its biggest markets.

The business question for this project: which customers stop buying, why, and how much revenue walks out with them?

## Data disclaimer

Every row in this project is synthetic. CartPadi is not a real company and has no relation to any existing brand; product brands are invented. The script `scripts/01_data_generation.py` is the provenance of the data. It runs on a fixed seed (42), so the same five CSV files come out every time. This notebook runs that script and shows what it produced.

## Data model

```mermaid
erDiagram
    customers ||--o{ orders : places
    customers ||--o{ sessions : "browses (when logged in)"
    orders ||--|{ order_items : contains
    products ||--o{ order_items : "sold as"
    customers {
        string customer_id PK
        date signup_date
        string state
        string acquisition_channel
        int is_churned
    }
    orders {
        string order_id PK
        string customer_id FK
        datetime order_date
        string order_status
        string payment_method
        float order_total_ngn
    }
    order_items {
        string item_id PK
        string order_id FK
        string product_id FK
        int quantity
        float unit_price_ngn
    }
    products {
        string product_id PK
        string category
        float base_price_ngn
    }
    sessions {
        string session_id PK
        string customer_id FK
        datetime session_date
        int converted
    }
```

Snapshot date is 31 December 2025. A customer is churned when they have placed no non-cancelled order in the 90 days up to the snapshot.

## How the generator works

1. Calendar. Each day gets a trading intensity: November 2.4, December 1.95, April 1.65, January 0.85 and about 1.0 for normal months, with a small weekend lift. Repeat-purchase gaps run on this calendar, so a 28-day gap in a normal month takes about 11 days in November.
2. Customers. 25,000 signups spread over 2024 and 2025, following a softer version of the same calendar plus 30% growth. Each gets a state (Lagos 43%, Abuja 14%), channel, device, preferred payment method and a favourite category.
3. Repeat odds. Each customer's chance of a second order within 90 days is the channel's target rate, scaled by payment method, signup quarter (Q4 deal-hunters 36% lower than Q1 and Q2 before other effects), region and favourite category. A bad first order (cancelled, returned or slow) cuts those odds.
4. Orders. Repeaters split into occasional buyers (two to four orders) and loyal buyers who keep a personal buying rhythm until they drop out. About four in ten repeaters are seasonal returners who only come back months later.
5. Order lines. One to four lines per order. The lead line usually comes from the favourite category; add-ons lean cheap. Prices rise about 28% over the two years and discounts peak in November.
6. Sessions. A 14.5% sample of orders gets a converted session; the rest of the 480,000 sessions are browsing by known customers (tailing off after their last order) and anonymous visitors.
7. Noise. About 0.15% of order rows are written twice and about 0.8% of customers lose their city, so the notebook 02 audit has real problems to find.

In [1]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().parent
result = subprocess.run([sys.executable, str(ROOT / "scripts" / "01_data_generation.py")],
                        capture_output=True, text=True, check=True)
print(result.stdout)

Building products ...
Building customers ...
Building orders ...
Building order items ...
Building sessions ...

Rows written
  customers       25,000
  products         2,000
  orders         121,208
  order_items    275,582
  sessions       480,000

Files in /home/user/cartpadi-customer/data/raw



In [2]:
import pandas as pd

RAW = ROOT / "data" / "raw"
for name in ["customers", "orders", "order_items", "products", "sessions"]:
    df = pd.read_csv(RAW / f"{name}.csv", nrows=5)
    print(f"{name}.csv")
    display(df)

customers.csv


,customer_id,signup_date,state,city,age_group,gender,acquisition_channel,first_device,is_churned
0,7b88783b-d1b4-4c94-8789-0e230e333911,2024-01-01,Plateau,Jos,25-34,M,referral,tablet,1
1,26673f00-641e-4cbd-acff-583fb9ac6c79,2024-01-01,Abuja,Wuse,18-24,M,paid_social,mobile,1
2,72082ab6-717a-4bd7-a60d-48045c2f714a,2024-01-01,Lagos,NaN,25-34,M,referral,mobile,1
3,5df84d65-10e4-4ed0-b9c0-1a780589899a,2024-01-01,Plateau,Jos,25-34,F,paid_social,mobile,1
4,3751bac3-9b8e-4181-8a6f-1beeffe3363f,2024-01-01,Lagos,Ojodu,25-34,M,organic_search,mobile,1


orders.csv


,order_id,customer_id,order_date,order_status,payment_method,order_total_ngn,discount_ngn,delivery_days,delivery_state
0,CP-0000001,72082ab6-717a-4bd7-a60d-48045c2f714a,2024-01-01 07:11:53,delivered,card,83600.0,0.0,5.0,Lagos
1,CP-0000002,610d4ef3-5f87-4b5b-947b-70b71bf51425,2024-01-01 10:24:55,delivered,card,41440.0,510.0,14.0,Kaduna
2,CP-0000003,23856148-426c-474b-81b1-6cad9fb2e04e,2024-01-01 11:52:05,cancelled,pay_on_delivery,697285.0,415.0,NaN,Lagos
3,CP-0000004,6367cd4a-92a9-41d9-a0d6-ac2465c7d3d9,2024-01-01 12:37:52,delivered,card,55550.0,0.0,2.0,Lagos
4,CP-0000005,7be42e45-012b-48a6-b405-994b6137e835,2024-01-01 14:18:02,delivered,card,36815.0,1135.0,12.0,Delta


order_items.csv


,item_id,order_id,product_id,quantity,unit_price_ngn,discount_pct
0,IT-00000001,CP-0000001,PRD-00585,1,15700.0,0.0
1,IT-00000002,CP-0000001,PRD-00780,3,8100.0,0.0
2,IT-00000003,CP-0000001,PRD-01891,1,25400.0,0.0
3,IT-00000004,CP-0000001,PRD-00464,2,9100.0,0.0
4,IT-00000005,CP-0000002,PRD-00400,1,16100.0,0.0


products.csv


,product_id,product_name,category,subcategory,brand,base_price_ngn
0,PRD-00001,Arewa Digital Laptop 14in X,Electronics,Laptops,Arewa Digital,518599.0
1,PRD-00002,Lumora Power Bank 30000mAh Air V1,Electronics,Power Banks,Lumora,36399.0
2,PRD-00003,Arewa Digital Smart TV 55in Neo,Electronics,Televisions,Arewa Digital,900899.0
3,PRD-00004,Voltani Ultrabook Max,Electronics,Laptops,Voltani,1794899.0
4,PRD-00005,Voltani Wireless Earbuds Pro,Electronics,Earphones,Voltani,28899.0


sessions.csv


,session_id,customer_id,session_date,channel,device,pages_viewed,duration_min,converted
0,SES-0000001,NaN,2024-01-01 00:44:39,organic_search,mobile,5,4.5,0
1,SES-0000002,NaN,2024-01-01 02:12:56,paid_social,mobile,1,1.1,0
2,SES-0000003,NaN,2024-01-01 06:29:52,influencer,desktop,5,2.9,0
3,SES-0000004,72082ab6-717a-4bd7-a60d-48045c2f714a,2024-01-01 07:00:53,email,desktop,15,11.0,1
4,SES-0000005,NaN,2024-01-01 07:08:03,paid_social,mobile,12,3.2,0


## Pattern check

`scripts/validate_patterns.py` measures each injected pattern on the files just written. The later notebooks rediscover these from the data; this check only confirms the generator did its job.

In [3]:
result = subprocess.run([sys.executable, str(ROOT / "scripts" / "validate_patterns.py")],
                        capture_output=True, text=True, check=True)
print(result.stdout)

Row counts (orders after de-duplication)
  customers       25,000
  orders         121,026
  order_items    275,582
  products         2,000
  sessions       480,000

Orders per customer (all statuses)
  1 order     55.9%   target 55%
  2-4 orders  22.5%   target 25%
  5+ orders   21.6%   target 20%

Median days to second order: 28   target 28

90-day retention overall:  30.4%

90-day retention by acquisition channel
  referral          41.9%   target 45%
  email             39.1%   target 40%
  direct            35.3%   target 38%
  organic_search    31.6%   target 32%
  influencer        23.3%   target -
  paid_social       19.0%   target 20%

90-day retention by first-order payment method
  CartPadi_wallet   42.2%   target 40%
  card              32.3%   target 28%
  bank_transfer     30.4%   target 25%
  USSD              28.0%   target 25%
  pay_on_delivery   22.0%   target 17%

Signup Q1-Q2 retention  33.1%, Q4  23.0%, Q4 is 31% lower   target 30%

Monthly orders index (normal mo